In [1]:
import pdfplumber
import pandas as pd
import numpy as np

In [2]:
with pdfplumber.open("../data/raw/ts/account-report-MKI-February-2026.pdf") as pdf:
    print(f"Total Pages in PDF: {len(pdf.pages)}")
    
    page1 = pdf.pages[0]
    lines = page1.extract_text_lines()

    fulltext = page1.extract_table()
    state_name = "unknown"
    month_name = "unknown"
    
    for line in lines:
        text = line["text"]
        
        # 1. Extract State Name
        if "Government of" in text:
            state_parts = text.split("of")
            if len(state_parts) > 1:
                state_name = state_parts[1].split("State")[0].strip()
        
        # 2. Extract Month Name
        # Looking for the line: "Monthly Key Indicators for the month of February 2026"
        if "month of" in text:
            month_parts = text.split("month of")
            if len(month_parts) > 1:
                month_name = month_parts[1].split("2026")[0].strip()
        
        # 3. Stop if both are found
        if state_name != "unknown" and month_name != "unknown":
            break

    # Extracting Table Data
    table_data = page1.extract_table()
    
    print(f"State: {state_name}")
    print(f"Month: {month_name}")

Total Pages in PDF: 9
State: Telangana
Month: February


In [4]:
clean_headers =[
    "Description",
    "Budget Estimates 2025-26",
    "Actuals up to Feb 2026",
    "Percentage: Current 2025-26",
    "Percentage: Corresponding 2024-25"
]

df = pd.DataFrame(fulltext[2:],columns=clean_headers)


df["State"] =state_name
df["month"]=month_name


df.head()

,Description,Budget Estimates 2025-26,Actuals up to Feb 2026,Percentage: Current 2025-26,Percentage: Corresponding 2024-25,State,month
0,1 REVENUE RECEIPTS,229720.63,154346.85,67.19,61.53,Telangana,February
1,a)Tax Revenue,175319.36,139100.81,79.34,75.46,Telangana,February
2,i)Goods and Service Tax,59704.59,48036.72,80.46,79.26,Telangana,February
3,ii)Stamps and Regstration,19087.26,13775.51,72.17,38.58,Telangana,February
4,iii)Land Revenue,11.20,0.54,4.84,7.39,Telangana,February


In [7]:
remove_row = [
    "1 REVENUE RECEIPTS",
    "2 CAPITAL RECEIPTS",
    "a)Tax Revenue",
    "3 TOTAL RECEIPTS",
    "7 TOTAL EXPENDITURE [4+5] (***)"
]
df = df[~df["Description"].isin(remove_row)]

tax_rev_list = [
    "i)Goods and Service Tax",
    "ii)Stamps and Regstration",
    "iii)Land Revenue",
    "iv)Sales Tax",
    "v)State Excise Duties",
    "vi)States Share of Union Taxes (*)",
    "vii)Other Taxes and duties ($)"
    ]

# Use .loc[rows, column] = value
df.loc[df["Description"].isin(tax_rev_list), "cat2"] = "Tax Revenue"

df.loc[~df["Description"].isin(tax_rev_list), "cat2"] = df["Description"].str[2:]

#1 REVENUE RECEIPTS
rev_rec_list =[
    "i)Goods and Service Tax",
    "ii)Stamps and Regstration",
    "iii)Land Revenue",
    "iv)Sales Tax",
    "v)State Excise Duties",
    "vi)States Share of Union Taxes (*)",
    "vii)Other Taxes and duties ($)",
    "b)Non-Tax Revenue",
    "c)Grant in aid and Contributions"
]

df.loc[df["Description"].isin(rev_rec_list),"cat1"] = "REVENUE RECEIPTS"



Cap_rec_list = [
    "a)Recovery of Loans and Advances",
    "b)Other Receipts",
    "c)Borrowings & Other Liabilities (Net ) (**)"
]

df.loc[df["Description"].isin(Cap_rec_list),"cat2"]="Capital Receipts"


print(rev_rec_list)

df=df.reset_index(drop=True)
df.head(50)

['i)Goods and Service Tax', 'ii)Stamps and Regstration', 'iii)Land Revenue', 'iv)Sales Tax', 'v)State Excise Duties', 'vi)States Share of Union Taxes (*)', 'vii)Other Taxes and duties ($)', 'b)Non-Tax Revenue', 'c)Grant in aid and Contributions']


,Description,Budget Estimates 2025-26,Actuals up to Feb 2026,Percentage: Current 2025-26,Percentage: Corresponding 2024-25,State,month,cat2,cat1
0,i)Goods and Service Tax,59704.59,48036.72,80.46,79.26,Telangana,February,Tax Revenue,REVENUE RECEIPTS
1,ii)Stamps and Regstration,19087.26,13775.51,72.17,38.58,Telangana,February,Tax Revenue,REVENUE RECEIPTS
2,iii)Land Revenue,11.20,0.54,4.84,7.39,Telangana,February,Tax Revenue,REVENUE RECEIPTS
3,iv)Sales Tax,37463.90,30628.68,81.76,87.41,Telangana,February,Tax Revenue,REVENUE RECEIPTS
4,v)State Excise Duties,27623.36,20454.14,74.05,65.61,Telangana,February,Tax Revenue,REVENUE RECEIPTS
5,vi)States Share of Union Taxes (*),21195.18,18788.79,88.65,93.79,Telangana,February,Tax Revenue,REVENUE RECEIPTS
6,vii)Other Taxes and duties ($),10233.87,7416.43,72.47,72.10,Telangana,February,Tax Revenue,REVENUE RECEIPTS
7,b)Non-Tax Revenue,31618.77,9105.39,28.80,17.25,Telangana,February,Non-Tax Revenue,REVENUE RECEIPTS
8,c)Grant in aid and Contributions,22782.50,6140.65,26.95,27.78,Telangana,February,Grant in aid and Contributions,REVENUE RECEIPTS
9,a)Recovery of Loans and Advances,1106.93,42.53,3.84,1.11,Telangana,February,Capital Receipts,NaN
